# 01 — Initial NOAA setup
Run manually for a new workspace. For the existing workspace, skip this notebook and use 03 then 02.
The fixed 2026 download is the project's bootstrap archive, not the daily updater.
Prerequisites: professor's `workspace.default.silver_station_info` table and the
`workspace.default.citibike` volume. No Git files or professor's tables are modified.

Existing bootstrap tables and registry settings are preserved. Rerunning this notebook
still checks NOAA and downloads files; it does not refresh existing candidate tables.
Use a reviewed migration for new bike stations or a changed study period.

## Download NOAA inventory

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

from pathlib import Path
import requests

# Keep original NOAA downloads in their own folder.
NOAA_FOLDER = Path("/Volumes/workspace/default/citibike/noaa")
NOAA_FOLDER.mkdir(parents=True, exist_ok=True)

STATION_LIST_URL = (
    "https://www.ncei.noaa.gov/oa/"
    "global-historical-climatology-network/hourly/doc/"
    "ghcnh-station-list.txt"
)

response = requests.get(STATION_LIST_URL, timeout=120)
response.raise_for_status()

# Catch an unexpected web page instead of a station-list download.
if "<html" in response.text[:1000].lower():
    raise ValueError("NOAA returned a web page instead of the station list.")

station_list_path = NOAA_FOLDER / "ghcnh-station-list.txt"
station_list_path.write_bytes(response.content)

print(f"Saved station inventory: {station_list_path}")
print(f"Downloaded {len(response.content):,} bytes")

# Inspect the format before splitting it into columns.
print("\nFirst 25 lines:\n")
for line in response.text.splitlines()[:25]:
    print(line)

## Parse inventory and select the broad region

In [0]:
from pathlib import Path
from pyspark.sql import functions as F

inventory_path = Path(
    "/Volumes/workspace/default/citibike/noaa/ghcnh-station-list.txt"
)

rows = []
unparsed_lines = []

for line in inventory_path.read_text().splitlines():
    if not line.strip():
        continue

    # The first four fields are ID, latitude, longitude, and elevation.
    # Keep the remaining text together because station names contain spaces.
    fields = line.split(maxsplit=4)

    try:
        station_id, lat, lon, elevation = fields[:4]
        rows.append((
            station_id,
            float(lat),
            float(lon),
            float(elevation),
            fields[4] if len(fields) > 4 else ""
        ))
    except ValueError:
        unparsed_lines.append(line)

if unparsed_lines:
    raise ValueError(f"Inventory has {len(unparsed_lines)} unparsed lines; inspect before continuing.")

noaa_stations = spark.createDataFrame(
    rows,
    """
    noaa_station_id STRING,
    noaa_lat DOUBLE,
    noaa_lon DOUBLE,
    elevation_m DOUBLE,
    station_description STRING
    """
)

# Broad search area around NYC, including nearby parts of NJ and NY.
noaa_candidates = noaa_stations.filter(
    F.col("noaa_lat").between(40.0, 42.0)
    & F.col("noaa_lon").between(-75.0, -72.0)
)

print("Inventory stations parsed:", len(rows))
print("Unparsed lines:", len(unparsed_lines))
print("Stations in our search area:", noaa_candidates.count())

if unparsed_lines:
    print("First unparsed lines:", unparsed_lines[:3])

display(noaa_candidates.orderBy("noaa_station_id"))

## Calculate candidate distances

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Get the latest metadata record for each Citi Bike station.
info = spark.table("workspace.default.silver_station_info")

latest = Window.partitionBy("station_id").orderBy(
    F.col("fetched_at").desc_nulls_last(),
    F.col("feed_updated_at").desc_nulls_last()
)

bike_stations = (
    info
    .filter(
        F.col("station_id").isNotNull()
        & (F.trim("station_id") != "")
    )
    .withColumn("_row", F.row_number().over(latest))
    .filter(F.col("_row") == 1)
    .filter(
        F.col("lat").between(40.0, 42.0)
        & F.col("lon").between(-75.0, -72.0)
        & ~F.coalesce(F.col("_has_invalid_coordinates"), F.lit(False))
        & ~F.coalesce(F.col("_has_parse_error"), F.lit(False))
    )
    .select(
        "station_id",
        F.col("name").alias("bike_station_name"),
        F.col("lat").alias("bike_lat"),
        F.col("lon").alias("bike_lon"),
        "snapshot_date"
    )
)

# 2. Keep inventory entries explicitly marked NY or NJ.
nearby_inventory = noaa_candidates.filter(
    F.trim(F.col("station_description")).rlike(r"^(NY|NJ)\s")
)

# 3. Compare each bike station with the small weather-station list.
pairs = bike_stations.crossJoin(F.broadcast(nearby_inventory))

# Great-circle distance using the Haversine formula.
lat1 = F.radians(F.col("bike_lat"))
lat2 = F.radians(F.col("noaa_lat"))
dlat = lat2 - lat1
dlon = F.radians(F.col("noaa_lon") - F.col("bike_lon"))

a = (
    F.sin(dlat / 2) ** 2
    + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) ** 2
)

within_50km = (
    pairs
    .withColumn(
        "distance_km",
        6371.0088 * 2 * F.asin(
            F.sqrt(F.greatest(F.lit(0.0), F.least(F.lit(1.0), a)))
        )
    )
    .filter(F.col("distance_km") <= 50)
)

# 4. Rank candidates for each bike station.
nearest_first = Window.partitionBy("station_id").orderBy(
    "distance_km", "noaa_station_id"
)

weather_candidates = within_50km.withColumn(
    "distance_rank",
    F.row_number().over(nearest_first)
)

print("Citi Bike stations with usable locations:", bike_stations.count())
print(
    "Distinct NOAA candidates within 50 km:",
    weather_candidates.select("noaa_station_id").distinct().count()
)
print(
    "Bike stations without a candidate:",
    bike_stations.select("station_id").join(
        weather_candidates.select("station_id").distinct(),
        "station_id",
        "left_anti"
    ).count()
)

display(
    weather_candidates
    .filter(F.col("distance_rank") <= 3)
    .select(
        "station_id",
        "bike_station_name",
        "noaa_station_id",
        "station_description",
        F.round("distance_km", 2).alias("distance_km"),
        "distance_rank"
    )
    .orderBy("station_id", "distance_rank")
    .limit(30)
)

## Check 2026 file availability

In [0]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

YEAR = 2026

candidate_ids = [
    row.noaa_station_id
    for row in weather_candidates
        .select("noaa_station_id")
        .distinct()
        .orderBy("noaa_station_id")
        .collect()
]

session = requests.Session()
session.mount(
    "https://",
    HTTPAdapter(max_retries=Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504]
    ))
)

checks = []

for station_id in candidate_ids:
    url = (
        "https://www.ncei.noaa.gov/oa/"
        "global-historical-climatology-network/hourly/access/"
        f"by-year/{YEAR}/parquet/GHCNh_{station_id}_{YEAR}.parquet"
    )

    try:
        response = session.head(url, timeout=30, allow_redirects=True)
        content_type = response.headers.get("Content-Type", "").lower()

        if response.status_code == 200 and "text/html" not in content_type:
            result = "available"
        elif response.status_code == 404:
            result = "not_found_at_url"
        else:
            result = f"needs_review_http_{response.status_code}"

        checks.append((station_id, YEAR, result, url))
        print(station_id, result)

    except requests.RequestException as error:
        checks.append((station_id, YEAR, "request_failed", url))
        print(station_id, "request_failed:", str(error)[:150])

weather_file_inventory = spark.createDataFrame(
    checks,
    """
    noaa_station_id STRING,
    year INT,
    file_status STRING,
    source_url STRING
    """
)

display(
    weather_file_inventory.groupBy("file_status").count()
)

## Download the nearest three available candidates

In [0]:
from pathlib import Path
import requests
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Match distance candidates to available files.
available_files = weather_file_inventory.filter(
    F.col("file_status") == "available"
)

rank_available = Window.partitionBy("station_id").orderBy(
    "distance_km", "noaa_station_id"
)

selected_candidates = (
    weather_candidates
    .join(available_files, "noaa_station_id", "inner")
    .withColumn("available_rank", F.row_number().over(rank_available))
    .filter(F.col("available_rank") <= 3)
)

unmatched = bike_stations.select("station_id").join(
    selected_candidates.select("station_id").distinct(),
    "station_id",
    "left_anti"
).count()

print("Bike stations without an available weather candidate:", unmatched)

files_to_download = (
    selected_candidates
    .select("noaa_station_id", "year", "source_url")
    .distinct()
    .orderBy("noaa_station_id")
    .collect()
)

download_folder = Path(
    "/Volumes/workspace/default/citibike/noaa/raw/2026"
)
download_folder.mkdir(parents=True, exist_ok=True)

print("Distinct weather files selected:", len(files_to_download))

downloaded_paths = []

for item in files_to_download:
    destination = download_folder / (
        f"GHCNh_{item.noaa_station_id}_{item.year}.parquet"
    )
    temporary = destination.with_suffix(".parquet.part")

    # Download to a temporary file so an interrupted transfer
    # isn't mistaken for a finished download.
    with session.get(
        item.source_url, stream=True, timeout=(15, 120)
    ) as response:
        response.raise_for_status()
        with temporary.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)

    if temporary.stat().st_size < 8:
        raise ValueError(f"Download too small: {temporary}")

    # Verify that the response looks like a Parquet file.
    with temporary.open("rb") as downloaded:
        header = downloaded.read(4)
        downloaded.seek(-4, 2)
        footer = downloaded.read(4)

    if header != b"PAR1" or footer != b"PAR1":
        raise ValueError(f"Unexpected file format: {temporary}")

    temporary.replace(destination)
    downloaded_paths.append(str(destination))
    print("Downloaded:", destination.name)

if not downloaded_paths:
    raise ValueError("No files selected; inspect the candidate matches.")

# Inspect one file before combining and transforming the observations.
sample_weather = spark.read.parquet(downloaded_paths[0])
sample_weather.printSchema()
display(sample_weather.limit(5))

## Inspect downloaded coverage

In [0]:
from pyspark.sql import functions as F

# Use the exact files downloaded in the preceding cell.
weather_raw = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(*downloaded_paths)
)

# DATE is retained as supplied by NOAA.
# Its ISO-formatted strings allow chronological min/max comparisons.
weather_coverage = (
    weather_raw
    .groupBy("STATION", "Station_name")
    .agg(
        F.count("*").alias("rows"),
        F.min("DATE").alias("first_observation"),
        F.max("DATE").alias("latest_observation"),
        F.count("temperature").alias("temperature_non_null"),
        F.count("wind_speed").alias("wind_speed_non_null"),
        F.count("precipitation").alias("precipitation_non_null")
    )
    .orderBy("STATION")
)

display(weather_coverage)

## Save bootstrap data and initialize the main raw table

In [0]:
from pyspark.sql import functions as F

# NOAA documents these observation timestamps as UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

weather_to_save = (
    weather_raw
    .withColumn(
        "observed_at_utc",
        F.expr("try_cast(`DATE` AS TIMESTAMP)")
    )
    # This records when we loaded this snapshot into the table.
    # It is NOT the original publication/availability time.
    .withColumn("snapshot_loaded_at_utc", F.current_timestamp())
    .withColumn(
        "source_url",
        F.concat(
            F.lit(
                "https://www.ncei.noaa.gov/oa/"
                "global-historical-climatology-network/hourly/"
                "access/by-year/2026/parquet/GHCNh_"
            ),
            F.col("STATION"),
            F.lit("_2026.parquet")
        )
    )
)

bad_timestamps = weather_to_save.filter(
    F.col("observed_at_utc").isNull()
).count()

if bad_timestamps:
    raise ValueError(
        f"{bad_timestamps} observations have missing/invalid timestamps."
    )

# Preserve existing bootstrap tables on repeat runs.
for table_name, frame in [
    ("workspace.default.noaa_weather_raw_2026", weather_to_save),
    ("workspace.default.citibike_noaa_candidates", weather_candidates),
]:
    if spark.catalog.tableExists(table_name):
        print("Preserving existing table:", table_name)
    else:
        frame.write.format("delta").mode("error").saveAsTable(table_name)

if not spark.catalog.tableExists("workspace.default.noaa_weather_raw"):
    spark.sql("""
        CREATE TABLE workspace.default.noaa_weather_raw USING DELTA
        AS SELECT * FROM workspace.default.noaa_weather_raw_2026
    """)

display(spark.sql("""
    SELECT
        STATION,
        COUNT(*) AS observations,
        MIN(observed_at_utc) AS first_observation,
        MAX(observed_at_utc) AS latest_observation
    FROM workspace.default.noaa_weather_raw_2026
    GROUP BY STATION
    ORDER BY STATION
"""))

## Define the observation view against the maintained table

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.default.noaa_weather_observations AS
SELECT
    STATION AS noaa_station_id,
    Station_name AS station_name,
    observed_at_utc,
    temperature,
    temperature_Quality_Code AS temperature_quality_code,
    temperature_Source_Code AS temperature_source_code,
    wind_speed,
    wind_speed_Quality_Code AS wind_quality_code,
    wind_speed_Source_Code AS wind_source_code,
    precipitation,
    precipitation_Measurement_Code AS precipitation_measurement_code,
    precipitation_Quality_Code AS precipitation_quality_code,
    precipitation_Source_Code AS precipitation_source_code,
    precipitation_Report_Type AS precipitation_report_type,
    CASE
        WHEN precipitation_Measurement_Code = 'T' THEN TRUE
        WHEN precipitation IS NOT NULL THEN FALSE
        ELSE NULL
    END AS precipitation_is_trace,
    CASE
        WHEN precipitation_Measurement_Code = 'T'
             OR precipitation > 0 THEN TRUE
        WHEN precipitation = 0 THEN FALSE
        ELSE NULL
    END AS precipitation_reported,
    source_url,
    snapshot_loaded_at_utc
FROM workspace.default.noaa_weather_raw
""")

## Initialize the station registry without changing existing choices

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.noaa_station_registry
USING DELTA
AS
SELECT
    STATION AS noaa_station_id,
    MAX(Station_name) AS station_name,
    TRUE AS download_enabled
FROM workspace.default.noaa_weather_raw
GROUP BY STATION
""")

# Show the saved station list and how recent each station's data is.
display(spark.sql("""
SELECT
    r.noaa_station_id,
    r.station_name,
    r.download_enabled,
    MAX(w.observed_at_utc) AS latest_observation_utc,
    ROUND(
        (
            unix_timestamp(current_timestamp())
            - unix_timestamp(MAX(w.observed_at_utc))
        ) / 3600.0,
        1
    ) AS hours_since_latest_observation
FROM workspace.default.noaa_station_registry r
LEFT JOIN workspace.default.noaa_weather_observations w
    ON r.noaa_station_id = w.noaa_station_id
GROUP BY
    r.noaa_station_id,
    r.station_name,
    r.download_enabled
ORDER BY r.noaa_station_id
"""))

Next: run 03 to create the mapping and derived views, then test 02 as a job. Do not schedule 01.